## UI 작업 
- Catalog > 개인 스키마 이동
- Volume 생성 (볼륨명: raw_data)
- Volume 안에 디렉토리 생성 (디렉토리명: parsed_images)
- 샘플 PDF 파일 "CustomerHandlingMaual.pdf"를 볼륨에 저장 



## ai_parse_document() 함수를 활용하여 PDF 문서 파싱
지원 포맷 
* PDF 
* JPG/JPEG
* PNG
* DOC/DOCX
* PPT/PPTX

In [0]:
%sql

-- To-Do: 개별 실습 환경에 맞게 카탈로그, 스키마, 볼륨 경로 변경 
CREATE or REPLACE TABLE hpark_demos.ski_agent_workshop.doc_parsed_tab 
AS SELECT
  path,
  ai_parse_document(
    content,
    map(
      'version', '2.0',
      'imageOutputPath', '/Volumes/hpark_demos/ski_agent_workshop/raw_data/parsed_images/',
      'descriptionElementTypes', '*'
    )
  ) as parsed_doc
FROM READ_FILES('/Volumes/hpark_demos/ski_agent_workshop/raw_data/*.pdf', format => 'binaryFile');



In [0]:
%sql

-- 파싱된 결과 확인
SELECT * FROM hpark_demos.ski_agent_workshop.doc_parsed_tab LIMIT 10

In [0]:
%sql

-- Vector Search 생성에 필요한 element만 추출하여 테이블 생성
-- To-Do: 카탈로그 및 스키마 명 변경 

CREATE OR REPLACE TABLE hpark_demos.ski_agent_workshop.doc_for_vector_search
TBLPROPERTIES (delta.enableChangeDataFeed = true) AS
WITH json_parsed AS (
  SELECT 
    path,
    explode(from_json(to_json(parsed_doc:document.elements), 'ARRAY<STRUCT<id:INT, content:STRING, bbox:ARRAY<STRUCT<page_id:INT, coord:ARRAY<INT>>>, type:STRING>>')) as element
  FROM hpark_demos.ski_agent_workshop.doc_parsed_tab
)
SELECT 
    path,
    element.content AS chunk_text,
    element.type AS element_type,
    element.bbox[0].page_id AS element_page_id,
    md5(concat(path, element.id)) as chunk_id
FROM json_parsed 
WHERE element.content IS NOT NULL AND element.type = 'text';

In [0]:
%sql

-- 저장된 결과 확인
SELECT * FROM hpark_demos.ski_agent_workshop.doc_for_vector_search LIMIT 10

### UI 작업: Vector Search endpoint 생성

* Compute > Vector Search > Create endpoint


### UI작업: Vector Search Index 생성 
- Catalog > Table > Create Vector search index
- 시간 소요 


### vector_search 함수를 통한 Vector Search Index 검색

- 참고자료  
https://docs.databricks.com/aws/en/sql/language-manual/functions/vector_search


In [0]:
%sql
SELECT * FROM vector_search(
  index => 'hpark_demos.ski_agent_workshop.doc_vector_index',
  query_text => '전화를 연결하는 방법',
  --query_text => '양해를 구하는 방법',
  query_type => 'ANN',
  num_results => 5)

In [0]:
%sql
SELECT * FROM vector_search(
  index => 'hpark_demos.ski_agent_workshop.doc_vector_index',
  query_text => '전화를 연결하는 방법',
  --query_text => '양해를 구하는 방법',
  query_type => 'HYBRID',
  num_results => 5)

### Reranker 활용하여 쿼리 품질 향상


In [0]:
%pip install databricks-vectorsearch --force-reinstall
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.reranker import DatabricksReranker
from databricks.vector_search.client import VectorSearchClient

# To-Do: 개별 환경에 맞게 수정하세요.
vector_search_endpoint_name = "ski_workshop_vector_index"
vs_index = f"hpark_demos.ski_agent_workshop.doc_vector_index"

vs_client = VectorSearchClient()
index = vs_client.get_index(endpoint_name=vector_search_endpoint_name, index_name=vs_index)

index.describe()



In [0]:
# reranker를 사용할 때는 최종적으로 반환할 컬럼(columns)과 reranker 과정에서 활용할 메타데이터 컬럼(columns_to_rerank)을 각각 별도로 설정합니다. 
# columns: 최종 결과 화면에 보여주고 싶은 정보(예: 제목, 본문 내용 등)입니다.
# columns_to_rerank: reranker 모델이 "질문과 얼마나 관련 있는지" 정밀하게 계산할 때 참고해야 할 텍스트 컬럼입니다. 주로 본문이나 핵심 키워드 컬럼을 지정합니다.

results = index.similarity_search(
    query_text = "전화를 연결하는 방법",
    columns = ["path", "element_type", "chunk_text"],
    num_results = 10,
    query_type = "hybrid",
    reranker=DatabricksReranker(columns_to_rerank=["chunk_text"])
    )


results

### 에이전트에서 활용 1: Playground에서 RAG 테스트
### UI 작업
- Playground 이동 
- 모델 선택
- Tool 선택


### 에이전트에서 활용 2: Tool-calling agent 에 Vector Search Index 추가